# DyTopo Embedding Analysis
**Paper**: Lu et al. 2026 — *DyTopo: Dynamic Topology for Multi-Agent Collaboration* (arXiv:2602.06039)

**Target runtime**: Google Colab G4 (NVIDIA RTX PRO 6000 · 96 GB VRAM)

This notebook probes the semantic geometry that drives DyTopo's topology-induction step:
how query/key sentence embeddings evolve across rounds and why τ = 0.3 produces the
dense → moderate → sparse adjacency pattern described in the paper.

In [ ]:
# Cell 1 — GPU Health Check
!nvidia-smi
import subprocess, sys
result = subprocess.run(['python', '-c', 'import torch; print("CUDA available:", torch.cuda.is_available()); print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("torch not yet installed — install in Cell 2 first")

In [ ]:
# Cell 2 — Install Dependencies
!pip install sentence-transformers httpx numpy matplotlib scikit-learn pandas nest_asyncio seaborn -q

In [ ]:
# Cell 3 — Download DyTopo Engine & Apply nest_asyncio
import os, sys, urllib.request

ENGINE_URL  = 'https://raw.githubusercontent.com/51103220/agentic-dynamic-topology/refs/heads/main/api/dytopo_engine.py'
ENGINE_PATH = '/content/dytopo_engine.py'

try:
    urllib.request.urlretrieve(ENGINE_URL, ENGINE_PATH)
    print(f'✓ Downloaded dytopo_engine.py ({os.path.getsize(ENGINE_PATH):,} bytes)')
except Exception as e:
    raise RuntimeError(
        f'Download failed: {e}\n'
        'Manually upload dytopo_engine.py to /content/ via the Files panel.'
    )

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import nest_asyncio
nest_asyncio.apply()
print('✓ nest_asyncio applied — asyncio.run() safe in Colab')

import importlib.util
spec = importlib.util.spec_from_file_location('dytopo_engine', ENGINE_PATH)
assert spec is not None, 'Cannot find engine file'
print('✓ Engine file found and importable')

In [ ]:
# Cell 4 — CUDA-Patched Imports & Q/K String Extraction
import torch
from sentence_transformers import SentenceTransformer

# Patch the module-level encoder to use CUDA before any encode() calls
import dytopo_engine
_cuda_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
dytopo_engine._encoder = _cuda_model
print(f"Encoder device: {next(_cuda_model.parameters()).device}")

# Now import public symbols
from dytopo_engine import (
    encode,
    _CODE_MOCK,
    _MATH_MOCK,
    CODE_ROLES,
    MATH_ROLES,
    build_adjacency,
)

# ── Extract Q and K strings per round from mock dicts ────────────────────────
def extract_qk(mock_dict, roles):
    """Returns {round_idx: {role: (q_str, k_str)}} for rounds 0,1,2."""
    data = {}
    for r in range(3):
        data[r] = {}
        for role in roles:
            entry = mock_dict.get((role, r), {})
            q = entry.get("q_vector", f"[{role} missing q r{r}]")
            k = entry.get("k_vector", f"[{role} missing k r{r}]")
            data[r][role] = (q, k)
    return data

code_qk  = extract_qk(_CODE_MOCK, CODE_ROLES)
math_qk  = extract_qk(_MATH_MOCK, MATH_ROLES)

print("\n=== CODE task Q/K strings per round ===")
for r in range(3):
    print(f"\n--- Round {r} ---")
    for role, (q, k) in code_qk[r].items():
        print(f"  {role:12s}  Q: {q}")
        print(f"  {role:12s}  K: {k}")

print("\n=== MATH task Q/K strings per round ===")
for r in range(3):
    print(f"\n--- Round {r} ---")
    for role, (q, k) in math_qk[r].items():
        print(f"  {role:15s}  Q: {q}")
        print(f"  {role:15s}  K: {k}")

In [ ]:
# Cell 5 — Compute Similarity Matrices R = Q @ K.T per Round
import numpy as np
import pandas as pd

TAU = 0.3

def compute_round_matrices(qk_data, roles):
    """For each round, returns (R, A) as np.ndarray."""
    R_list, A_list = [], []
    for r in range(3):
        queries = [qk_data[r][role][0] for role in roles]
        keys    = [qk_data[r][role][1] for role in roles]
        Q = encode(queries)          # (N, 384) L2-normalised
        K = encode(keys)
        R = Q @ K.T                  # (N, N) cosine similarities
        A = (R > TAU).astype(int)
        np.fill_diagonal(A, 0)
        R_list.append(R)
        A_list.append(A)
    return R_list, A_list

print("Encoding CODE task embeddings on CUDA...")
code_R_list, code_A_list = compute_round_matrices(code_qk, CODE_ROLES)

print("Encoding MATH task embeddings on CUDA...")
math_R_list, math_A_list = compute_round_matrices(math_qk, MATH_ROLES)

# ── Summary statistics ─────────────────────────────────────────────────────
print("\n=== Mean Cosine Similarity per Round ===")
print(f"{'Task':<8} {'Round':<8} {'Mean cosine':>12} {'Off-diag mean':>15} {'Density (τ=0.3)':>16}")
print("-" * 62)

for task_name, R_list, A_list in [("CODE", code_R_list, code_A_list), ("MATH", math_R_list, math_A_list)]:
    N = R_list[0].shape[0]
    for r in range(3):
        R = R_list[r]
        A = A_list[r]
        mask = ~np.eye(N, dtype=bool)
        mean_cos   = R[mask].mean()
        mean_all   = R.mean()
        density    = A[mask].mean()
        print(f"{task_name:<8} {r:<8} {mean_all:>12.4f} {mean_cos:>15.4f} {density:>16.3f}")

In [ ]:
# Cell 6 — 2×3 Heatmap Grid: Raw R (top) + Binary A (bottom)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "figure.facecolor":  "#0e1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#30363d",
    "axes.labelcolor":   "#c9d1d9",
    "xtick.color":       "#8b949e",
    "ytick.color":       "#8b949e",
    "text.color":        "#c9d1d9",
    "grid.color":        "#21262d",
    "axes.titlecolor":   "#e6edf3",
    "font.family":       "DejaVu Sans",
    "font.size":         10,
})

def plot_heatmap_grid(R_list, A_list, roles, task_name):
    N = len(roles)
    fig, axes = plt.subplots(2, 3, figsize=(14, 9),
                              facecolor="#0e1117", constrained_layout=True)
    fig.suptitle(f"DyTopo Embedding Analysis — {task_name} Task\n"
                 f"Top row: raw cosine similarity R = Q·Kᵀ  |  "
                 f"Bottom row: binary adjacency A (τ = {TAU})",
                 fontsize=13, color="#e6edf3", y=1.01)

    for r in range(3):
        R = R_list[r]
        A = A_list[r]
        mask = ~np.eye(N, dtype=bool)
        density = A[mask].mean() * 100

        # ── Top row: R matrix ──────────────────────────────────────────────
        ax_r = axes[0, r]
        im_r = ax_r.imshow(R, cmap="RdYlGn", vmin=-1, vmax=1,
                            aspect="auto", interpolation="nearest")
        ax_r.set_title(f"Round {r} — R matrix", fontsize=11, color="#e6edf3", pad=6)
        ax_r.set_xticks(range(N))
        ax_r.set_yticks(range(N))
        ax_r.set_xticklabels(roles, rotation=35, ha="right", fontsize=8)
        ax_r.set_yticklabels(roles, fontsize=8)
        ax_r.set_xlabel("Key agent (j)", fontsize=8)
        ax_r.set_ylabel("Query agent (i)", fontsize=8)
        # Annotate cell values
        for i in range(N):
            for j in range(N):
                val = R[i, j]
                col = "#0e1117" if abs(val) > 0.4 else "#c9d1d9"
                ax_r.text(j, i, f"{val:.2f}", ha="center", va="center",
                          fontsize=7, color=col)
        cb_r = fig.colorbar(im_r, ax=ax_r, shrink=0.8, pad=0.02)
        cb_r.ax.tick_params(colors="#8b949e", labelsize=7)
        cb_r.outline.set_edgecolor("#30363d")

        # ── Bottom row: A matrix ───────────────────────────────────────────
        ax_a = axes[1, r]
        im_a = ax_a.imshow(A, cmap="Blues", vmin=0, vmax=1,
                            aspect="auto", interpolation="nearest")
        ax_a.set_title(f"Round {r} — A matrix  ({density:.0f}% density)",
                        fontsize=11, color="#e6edf3", pad=6)
        ax_a.set_xticks(range(N))
        ax_a.set_yticks(range(N))
        ax_a.set_xticklabels(roles, rotation=35, ha="right", fontsize=8)
        ax_a.set_yticklabels(roles, fontsize=8)
        ax_a.set_xlabel("Source agent j", fontsize=8)
        ax_a.set_ylabel("Target agent i", fontsize=8)
        for i in range(N):
            for j in range(N):
                lbl = "1" if A[i, j] else "·"
                col = "#e6edf3" if A[i, j] == 0 else "#0e1117"
                ax_a.text(j, i, lbl, ha="center", va="center",
                          fontsize=9, color=col, fontweight="bold")
        cb_a = fig.colorbar(im_a, ax=ax_a, shrink=0.8, pad=0.02,
                             ticks=[0, 1])
        cb_a.ax.set_yticklabels(["0 (no edge)", "1 (edge)"], fontsize=7)
        cb_a.ax.tick_params(colors="#8b949e")
        cb_a.outline.set_edgecolor("#30363d")

    plt.savefig(f"dytopo_heatmaps_{task_name.lower()}.png", dpi=150,
                bbox_inches="tight", facecolor="#0e1117")
    plt.show()
    print(f"Saved dytopo_heatmaps_{task_name.lower()}.png")

plot_heatmap_grid(code_R_list, code_A_list, CODE_ROLES, "CODE")
plot_heatmap_grid(math_R_list, math_A_list, MATH_ROLES, "MATH")

In [ ]:
# Cell 7 — Histograms of Off-Diagonal Cosine Similarities per Round
ROUND_COLORS = ["#58a6ff", "#3fb950", "#f78166"]   # blue, green, red-orange
ROUND_LABELS = ["Round 0", "Round 1", "Round 2"]

def plot_cosine_histograms(R_list, task_name):
    N = R_list[0].shape[0]
    mask = ~np.eye(N, dtype=bool)

    fig, ax = plt.subplots(figsize=(9, 5), facecolor="#0e1117")
    ax.set_facecolor("#161b22")

    bins = np.linspace(-0.1, 1.0, 28)
    for r, (R, color, label) in enumerate(zip(R_list, ROUND_COLORS, ROUND_LABELS)):
        off_diag = R[mask]
        ax.hist(off_diag, bins=bins, alpha=0.55, color=color, label=label,
                edgecolor="#0e1117", linewidth=0.4)
        mean_val = off_diag.mean()
        ax.axvline(mean_val, color=color, linestyle="--", linewidth=1.2, alpha=0.85)
        ax.text(mean_val + 0.01, ax.get_ylim()[1] * 0.05, f"μ={mean_val:.3f}",
                color=color, fontsize=8, va="bottom", ha="left")

    # τ threshold line
    ax.axvline(TAU, color="#f0e68c", linestyle="-", linewidth=2.0,
               label=f"τ = {TAU} (threshold)", zorder=5)
    ax.text(TAU + 0.01, ax.get_ylim()[1] * 0.80, f"τ = {TAU}",
            color="#f0e68c", fontsize=9, fontweight="bold")

    ax.set_xlabel("Cosine similarity  q̂ᵢ · k̂ⱼ  (off-diagonal)", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)
    ax.set_title(f"Off-diagonal Cosine Similarities — {task_name} Task\n"
                 f"Values > τ activate edges in adjacency A",
                 fontsize=12, color="#e6edf3")
    ax.legend(framealpha=0.3, facecolor="#21262d", edgecolor="#30363d",
              labelcolor="#c9d1d9", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")

    plt.tight_layout()
    plt.savefig(f"dytopo_hist_{task_name.lower()}.png", dpi=150,
                bbox_inches="tight", facecolor="#0e1117")
    plt.show()
    print(f"Saved dytopo_hist_{task_name.lower()}.png")

plot_cosine_histograms(code_R_list, "CODE")
plot_cosine_histograms(math_R_list, "MATH")

In [ ]:
# Cell 8 — PCA Projection of All Q and K Embeddings Across Rounds
from sklearn.decomposition import PCA

def collect_all_embeddings(qk_data, roles):
    """Return (all_vecs, labels) where labels = list of (round, role, kind)."""
    vecs, labels = [], []
    for r in range(3):
        for role in roles:
            q_str, k_str = qk_data[r][role]
            vecs.append(encode([q_str])[0])
            labels.append((r, role, "Q"))
            vecs.append(encode([k_str])[0])
            labels.append((r, role, "K"))
    return np.array(vecs), labels

print("Encoding all embeddings for PCA...")
code_vecs, code_labels = collect_all_embeddings(code_qk, CODE_ROLES)
math_vecs, math_labels = collect_all_embeddings(math_qk, MATH_ROLES)

def plot_pca(vecs, labels, roles, task_name):
    pca = PCA(n_components=2, random_state=42)
    proj = pca.fit_transform(vecs)
    var  = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(11, 8), facecolor="#0e1117")
    ax.set_facecolor("#161b22")

    # Group by (round, role) to draw connecting lines between Q and K
    # Build a lookup: (round, role, kind) -> 2D point
    coord_map = {}
    for i, (r, role, kind) in enumerate(labels):
        coord_map[(r, role, kind)] = proj[i]

    # Draw Q-K connecting lines first (underneath points)
    for r in range(3):
        alpha = 0.25 + r * 0.12
        color = ROUND_COLORS[r]
        for role in roles:
            pq = coord_map.get((r, role, "Q"))
            pk = coord_map.get((r, role, "K"))
            if pq is not None and pk is not None:
                ax.plot([pq[0], pk[0]], [pq[1], pk[1]],
                        color=color, linewidth=1.1, alpha=alpha, zorder=1)

    # Plot Q points (circles) and K points (triangles)
    for i, (r, role, kind) in enumerate(labels):
        x, y   = proj[i]
        color  = ROUND_COLORS[r]
        marker = "o" if kind == "Q" else "^"
        size   = 90  if kind == "Q" else 100
        edge   = "#e6edf3" if kind == "Q" else "#f0e68c"
        ax.scatter(x, y, c=color, marker=marker, s=size, zorder=3,
                   edgecolors=edge, linewidths=0.8, alpha=0.9)
        # Label with role abbreviation
        short = role[:3]
        ax.annotate(short, (x, y), textcoords="offset points",
                    xytext=(5, 4), fontsize=7, color=color, alpha=0.85)

    # Legend patches
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        *[Patch(facecolor=ROUND_COLORS[r], label=f"Round {r}") for r in range(3)],
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#8b949e",
               markersize=8, label="Q (query)"),
        Line2D([0], [0], marker="^", color="w", markerfacecolor="#8b949e",
               markersize=8, label="K (key)"),
    ]
    ax.legend(handles=legend_elements, loc="upper left",
              framealpha=0.3, facecolor="#21262d", edgecolor="#30363d",
              labelcolor="#c9d1d9", fontsize=9)

    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}% var)", fontsize=11)
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}% var)", fontsize=11)
    ax.set_title(f"PCA of Q & K Embeddings — {task_name} Task\n"
                 f"Circles = Q (query), Triangles = K (key); lines connect Q-K pairs per role",
                 fontsize=12, color="#e6edf3")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")

    plt.tight_layout()
    plt.savefig(f"dytopo_pca_{task_name.lower()}.png", dpi=150,
                bbox_inches="tight", facecolor="#0e1117")
    plt.show()
    print(f"Saved dytopo_pca_{task_name.lower()}.png")

plot_pca(code_vecs, code_labels, CODE_ROLES, "CODE")
plot_pca(math_vecs, math_labels, MATH_ROLES, "MATH")

In [ ]:
# Cell 9 — Summary Table + Analysis
import pandas as pd

rows = []
for task_name, R_list, A_list, roles in [
    ("CODE", code_R_list, code_A_list, CODE_ROLES),
    ("MATH", math_R_list, math_A_list, MATH_ROLES),
]:
    N = len(roles)
    mask = ~np.eye(N, dtype=bool)
    for r in range(3):
        R = R_list[r]
        A = A_list[r]
        off = R[mask]
        density_pct   = A[mask].mean() * 100
        total_edges   = int(A[mask].sum())
        max_possible  = N * (N - 1)
        rows.append({
            "Task":          task_name,
            "Round":         r,
            "Roles":         ", ".join(roles),
            "Mean cos sim":  round(float(off.mean()), 4),
            "Std cos sim":   round(float(off.std()),  4),
            "Min cos sim":   round(float(off.min()),  4),
            "Max cos sim":   round(float(off.max()),  4),
            "Active edges":  total_edges,
            "Max edges":     max_possible,
            "Density %":     round(density_pct, 1),
            "Pattern":       "dense" if density_pct > 60 else "moderate" if density_pct > 25 else "sparse",
        })

df = pd.DataFrame(rows)

# Display with styling
try:
    from IPython.display import display
    styled = (
        df.style
          .background_gradient(subset=["Mean cos sim", "Density %"], cmap="RdYlGn")
          .set_caption("DyTopo Embedding Analysis — Summary Table")
          .format({"Mean cos sim": "{:.4f}", "Std cos sim": "{:.4f}",
                   "Min cos sim": "{:.4f}", "Max cos sim": "{:.4f}",
                   "Density %": "{:.1f}"})
    )
    display(styled)
except Exception:
    print(df.to_string(index=False))

print()
print("=" * 70)
print("WHY DO MOCK STRINGS PRODUCE A DENSE → SPARSE PATTERN?")
print("=" * 70)
analysis = """
The mock Q/K strings in dytopo_engine.py are deliberately crafted so that
cosine similarities between Q and K vectors evolve across rounds:

ROUND 0 — DENSE (high-density adjacency)
  • All agents begin with broad, cross-cutting queries:
    "I need algorithm analysis", "I need the API design", etc.
  • Keys describe general role expertise: "I provide complete implementation",
    "I provide algorithm analysis and time complexity".
  • The all-MiniLM-L6-v2 encoder maps these semantically overlapping phrases
    into a compact region of the 384-dim sphere, so most off-diagonal
    R[i,j] values exceed τ = 0.3 → many edges activate.

ROUND 1 — MODERATE (medium-density adjacency)
  • Queries become more specific, e.g. "I need the Tester to verify my
    implementation handles edge cases" (Developer) vs. "I need confirmation
    that tests pass" (Researcher).
  • Keys also specialise: "I provide complete verified Python implementation"
    is closer in embedding space to Developer-related queries but not to
    Designer queries.
  • The embedding neighbourhood shrinks → moderate density.

ROUND 2 — SPARSE (low-density adjacency)
  • Multiple agents declare "I need nothing further" — a convergent,
    near-zero-information query that the model places far from any content key.
  • Remaining queries (e.g. "I need Manager's final assessment") are highly
    role-specific, matching only one or two keys at best.
  • Result: very few off-diagonal similarities exceed τ → sparse A.

This mirrors Figure 4 of Lu et al. 2026: early rounds require global
information exchange (dense graph); as agents converge, communication
becomes targeted (sparse graph), reducing computational overhead while
maintaining task-relevant information flow.

τ = 0.3 is the paper's recommended threshold — low enough to allow early
broad connectivity, high enough to prune noise in later rounds.
"""
print(analysis)